# Module 3 · De l'inventaire au stock de carbone

**Objectif :** produire les indicateurs qu'on attend d'un inventaire (G, N, essences dominantes) puis estimer la biomasse et le carbone, comme dans un suivi REDD+.

**Durée indicative :** 3 à 4 heures. **Prérequis :** module 2b (numpy, fusion, graphiques).

**Nouveauté :** ici, tu ne fais plus de boucles. Tu travailles sur des colonnes entières, ce qui est plus rapide à écrire et à exécuter.

In [ ]:
# Cellule de démarrage : exécute-la en premier (clic sur ▶ ou Maj + Entrée)
import os, sys, subprocess
if not os.path.exists("outils/verif.py"):
    if "google.colab" in sys.modules:  # Colab : on récupère les données et les vérifications
        if not os.path.exists("/content/data-foret-depot"):
            subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "claude/quirky-keller-jhyoi0",
                            "https://github.com/Cas1233128/Cr-ation_page_web", "/content/data-foret-depot"], check=True)
        os.chdir("/content/data-foret-depot/data-foret")
    else:
        raise FileNotFoundError("Le dossier 'outils' est introuvable. Ouvre ce notebook depuis le dossier data-foret (voir le README).")
from outils.verif import verifier, indice
print("Prêt. Pour chaque exercice : écris ton code, exécute, lis le verdict.")

### Point de départ

Charge ton inventaire propre. Si tu n'as pas terminé le module 2, utilise `donnees/inventaire_propre_reference.csv`.

In [ ]:
import pandas as pd
import numpy as np
import os

fichier = "donnees/inventaire_propre.csv" if os.path.exists("donnees/inventaire_propre.csv") else "donnees/inventaire_propre_reference.csv"
df = pd.read_csv(fichier)
SURFACE_PLACETTE = 0.5  # ha
print(fichier, df.shape)

### Exercice 1 · Surface terrière de tous les arbres d'un coup

Au module 1 tu calculais *g* arbre par arbre avec une boucle. Avec pandas, une opération sur une colonne s'applique à **toutes les lignes en même temps**. Ajoute une colonne `g` (en m²).

In [ ]:
# À toi :


verifier("m3_e1", df)

### Exercice 2 · Surface terrière à l'hectare (G)

Calcule `G_ha` : la surface terrière totale **par placette**, ramenée à l'hectare (m²/ha). En forêt dense humide, on attend en général 20 à 35 m²/ha.

In [ ]:
# À toi :
G_ha = 
G_ha

verifier("m3_e2", G_ha)

### Exercice 3 · Densité de tiges (N)

Calcule `N_ha`, le nombre d'arbres par hectare pour chaque placette.

> Outil nouveau : `df.groupby("placette").size()` compte le nombre de lignes de chaque groupe (ici, d'arbres par placette).

In [ ]:
# À toi :
N_ha = 

verifier("m3_e3", N_ha)

### Exercice 4 · Essences dominantes

Quelles sont les **3 essences** qui pèsent le plus en surface terrière sur l'ensemble de l'inventaire ? Range leurs noms, de la plus importante à la moins importante, dans la liste `top3`.

In [ ]:
# À toi :
top3 = 

verifier("m3_e4", top3)

### Exercice 5 · Estimer les hauteurs manquantes

Pour chaque arbre sans hauteur, utilise la **médiane des hauteurs mesurées dans sa classe de diamètre** (classes de 10 cm, plafonnées à 120, comme au module 1). Crée d'abord une colonne `classe`, puis remplis `hauteur`.

> On utilise la médiane, pas la moyenne générale, car la hauteur dépend fortement du diamètre et la médiane résiste mieux aux erreurs de mesure.

In [ ]:
# À toi :


verifier("m3_e5", df)

### Exercice 6 · Ajouter la densité du bois

Le fichier `donnees/especes.csv` donne la densité du bois de chaque essence (valeurs **indicatives** ; pour un vrai travail, utilise la *Global Wood Density Database*). Fusionne-le avec `df`.

In [ ]:
# À toi :


verifier("m3_e6", df)

### Exercice 7 · L'équation allométrique de Chave (2014)

Pour estimer la biomasse aérienne d'un arbre tropical sans l'abattre, on utilise l'équation pantropicale de **Chave et al. (2014)** :

$$AGB = 0{,}0673 \times (\rho \times D^2 \times H)^{0{,}976}$$

avec ρ la densité du bois (g/cm³), D le DHP (cm), H la hauteur (m). AGB est en **kg**. Écris `biomasse_chave(densite, dhp, hauteur)`.

In [ ]:
# À toi :
def biomasse_chave(densite, dhp, hauteur):
    ...

verifier("m3_e7", biomasse_chave)

### Exercice 8 · Biomasse à l'hectare

Calcule la biomasse de chaque arbre, puis `AGB_ha` : la biomasse aérienne par placette en **tonnes par hectare**.

In [ ]:
# À toi :
AGB_ha = 
AGB_ha.describe()

verifier("m3_e8", AGB_ha)

### Exercice 9 · Du bois au carbone

Calcule `C_ha`, le stock de carbone (t C/ha) de chaque placette, puis `placette_max`, la placette qui en stocke le plus.

Bonus : en multipliant par 44/12, tu obtiens des tonnes d'équivalent CO₂, l'unité utilisée dans les projets REDD+.

In [ ]:
# À toi :
C_ha = 
placette_max = 

verifier("m3_e9", C_ha, placette_max)

### Exercice 10 · Lire ses résultats (sans vérification automatique)

Le graphique ci-dessous est fourni. Lance-le, puis réponds par écrit dans la cellule suivante :

1. La placette la plus riche en carbone est-elle aussi celle qui a le plus d'arbres ? Pourquoi ?
2. Regarde la composition en essences de cette placette (`df[df["placette"] == placette_max]["essence"].value_counts()`). Quel rôle joue la densité du bois ?
3. Quelle est la principale source d'incertitude dans ce calcul ? (indice : quelle variable avons-nous estimée ?)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
C_ha.sort_values().plot.barh(ax=axes[0], color="#2e7651")
axes[0].set_xlabel("Carbone (t C/ha)")
axes[0].set_ylabel("Placette")
axes[1].scatter(N_ha, C_ha, color="#2e7651")
axes[1].set_xlabel("Tiges par hectare")
axes[1].set_ylabel("Carbone (t C/ha)")
plt.tight_layout()
plt.show()

*Tes réponses :*

1. 
2. 
3. 

## Bilan du module 3

Tu as reproduit la chaîne de calcul d'un inventaire carbone : nettoyage, indicateurs dendrométriques, imputation, allométrie, agrégation.

**Ce qui vient ensuite :** les modules 4 à 7 (voir le README) traitent les statistiques, la modélisation, les pièges photographiques et la perte de couvert forestier.